# 10 — What are SPP1-TAMs *doing* in each condition? Pseudobulk DE → GSEA

### Was this already done? Partly — and the gaps are exactly the interesting ones.

`09` §8b ran pseudobulk DESeq2 on SPP1-TAMs, but only **two** contrast families, both with tumour
site collapsed into a covariate (`~ tu + cond`):

| contrast | in 09? | here? |
|---|---|---|
| RT vs matched non-RT, **sites pooled** | yes | — |
| treated vs NoTx (benchmark) | yes | kept, as a detection check only |
| **RT at the irradiated site only** (`Tu1`) | **no** | §5 tier A |
| **RT at the abscopal site only** (`Tu2`) | **no** | §5 tier A |
| **within-mouse Tu1 vs Tu2**, paired, per arm | **no** (only at score level) | §5 tier B |
| **difference-in-differences** (RT's effect on the Tu1−Tu2 gap) | **no** | §5 tier C |
| **CAR axis** (PSCA vs TAG72) | **no** | §5 tier D |
| enrichment by **GSEA NES** | **no** — 09 used a Mann-Whitney rank-sum `z` | §4, §7 |

That last row matters. Per the 2026-09-18 finding, the rank-sum `z` is anti-conservative (it
assumes gene independence) and its bar length scales with set size, so it is not comparable across
pathways. It also **missed the strongest result** in 01c: scavenger-receptor uptake had `z = 0.77`
but `NES = 1.97, FDR < 0.001`, because its signal is a few genes at the very top of the ranking —
which weighted-KS finds and a mean-rank shift does not. This notebook uses the validated inline
GSEA from `01c` §4d throughout.

### What the design will and will not support

- **No RT-alone arm.** Every contrast is *"adding RT on top of Cy + CAR-T"*.
- **n = 3 mice per arm**, 2 tumours each. Site-resolved contrasts are therefore **3 vs 3**
  (3 vs 2 for `CyTAG72_Tu1`). These are underpowered by construction; the evidence is
  **sign agreement across the two CAR-T backgrounds**, never a single p-value.
- **`NoTx` is not a biological comparator** — tumour is 25–42% of its cells vs 4–6% in treated
  arms. It appears here *only* to prove the pipeline can detect a known perturbation.
- **The dominant compositional axis in this dataset is which CAR was given, not radiation**
  (`rtAdvantage/02b`: CAR 10.3% of variance vs RT main effect 3.5%; PERMANOVA CAR p = 0.021,
  RT p = 0.597). Tier D measures that axis explicitly rather than letting it leak into the RT
  contrasts.
- **Expect the RT contrasts to be null between animals.** Every between-animal RT test in this
  project has been. The two places structure has appeared are the *within-mouse* pairing
  (`rtAdvantage/01`) and *spatial range* (`08b`). Tier B and C are the within-mouse versions here.

Helpers are inline — the HPC kernel cannot see local `.py` files.

## 1. Config

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, scanpy as sc, anndata as ad, h5py, scipy.sparse as sp
from scipy import stats
from statsmodels.stats.multitest import multipletests
from pathlib import Path
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

_HPC, _LOC = Path("/coh_labs/yunroseli/Jona/CAR-T"), Path("/Users/janzules/Roselab/Spatial/CAR_T")
PROJ = _HPC if _HPC.exists() else _LOC
SC_REF  = (PROJ/"data/zarr/fullDataset/annotating_references/annotation_v4.h5ad" if PROJ is _HPC
           else PROJ/"data/sc-reference/annotation_v4.h5ad")
MAC_REF = SC_REF.parent/"mac_annotation_v5.h5ad"
GMT_DIR = PROJ/"data/references/pathways"
SIG_CSV = PROJ/"results/spp1_analysis/sc_spp1_tam_signature.csv"
OUT     = PROJ/"results/spp1_analysis/sc_spp1tam_conditions"
for s in ["de","gsea","figures"]: (OUT/s).mkdir(parents=True, exist_ok=True)

MAC_LABELS = ["M1_like_Mac","M2_like_Mac","Intermediate_Mac"]
MIN_CELLS  = 20        # per-sample floor for a pseudobulk
MIN_SAMPLES, MIN_COUNT = 3, 10
N_PERM, SEED = 1000, 0
pd.set_option("display.width", 230, "display.max_columns", 60)
print(f"project : {PROJ}\noutputs : {OUT}")

project : /Users/janzules/Roselab/Spatial/CAR_T
outputs : /Users/janzules/Roselab/Spatial/CAR_T/results/spp1_analysis/sc_spp1tam_conditions


## 2. Load, design, and the SPP1-TAM population (same four definitions as 09)

In [2]:
with h5py.File(SC_REF,"r") as f:
    obs = ad.io.read_elem(f["obs"]); var = ad.io.read_elem(f["var"])
    X   = ad.io.read_elem(f["X"]); Craw = ad.io.read_elem(f["layers/counts"])
adata = ad.AnnData(X=X, obs=obs, var=var); del X
C_counts = sp.csr_matrix(Craw); del Craw
assert adata.X[:2000].max() < 50, ".X is not log-normalised"
print(f"{adata.n_obs:,} cells x {adata.n_vars:,} genes | counts layer integer = "
      f"{float(C_counts[:1000].data.max())==int(C_counts[:1000].data.max())}")

o = adata.obs
o["cond"]  = o["condition"].astype(str); o["sample"] = o["file_name"].astype(str)
o["tu"]    = o["sample"].str.extract(r"_(Tu\d)_")[0].values      # Tu1 irradiated, Tu2 abscopal
o["mouse"] = o["sample"].str.replace(r"_Tu\d_","_",regex=True)
o["ct"]    = o["cell_type_lvl2"].astype(str)
o["is_mac"]= o["ct"].isin(MAC_LABELS); o["is_M2"] = o["ct"]=="M2_like_Mac"
o["cart"]  = np.where(o["cond"].str.contains("PSCA"), "PSCA",
              np.where(o["cond"].str.contains("TAG72"), "TAG72", "none"))

sig = [g for g in pd.read_csv(SIG_CSV)["gene"] if g in adata.var_names]
sc.tl.score_genes(adata, sig, score_name="SPP1_TAM_sig", ctrl_size=50, random_state=0)
mac, s = o["is_mac"].values, adata.obs["SPP1_TAM_sig"].values
o["TAM_global"] = mac & (s >= np.quantile(s[mac], 0.75))
with h5py.File(MAC_REF,"r") as f: m_obs = ad.io.read_elem(f["obs"])
o["TAM_alice"] = o.index.map(m_obs["cell_type_lvl4"].astype(str).reindex(o.index)).fillna("NA").values=="Spp1_pos_mac"

print(f"\nSPP1-TAM (signature top quartile): {o.TAM_global.sum():,}")
t = o[o.TAM_global].groupby(["cond","tu"], observed=True).agg(
        n_samples=("sample","nunique"), n_cells=("sample","size"))
t["cells_per_sample"] = (t.n_cells/t.n_samples).round(0)
print(t.to_string())
print(f"\nsmallest per-sample SPP1-TAM pseudobulk: "
      f"{o[o.TAM_global].groupby('sample', observed=True).size().min()} cells  <- leverage check in §8")

93,548 cells x 22,949 genes | counts layer integer = True



SPP1-TAM (signature top quartile): 12,119
               n_samples  n_cells  cells_per_sample
cond      tu                                       
CyPSCA    Tu1          3      971             324.0
          Tu2          3      372             124.0
CyTAG72   Tu1          2     2217            1108.0
          Tu2          3     1712             571.0
NoTx      Tu1          3      527             176.0
          Tu2          3      261              87.0
RTCyPSCA  Tu1          3     1202             401.0
          Tu2          3      735             245.0
RTCyTAG72 Tu1          3     2751             917.0
          Tu2          3     1371             457.0

smallest per-sample SPP1-TAM pseudobulk: 26 cells  <- leverage check in §8


## 3. Pseudobulk — summed raw counts, one observation per sample

Summing counts (not averaging per-cell log values) is what makes a sample the unit and lets
DESeq2's median-of-ratios size factors normalise depth at the level the test operates on.

In [3]:
def pseudobulk(mask, min_cells=MIN_CELLS):
    d = adata.obs[mask]; n = d.groupby("sample", observed=True).size()
    ok = list(n[n >= min_cells].index)
    idx = np.where(mask & adata.obs["sample"].isin(ok).values)[0]
    sub, ot = C_counts[idx], adata.obs.iloc[idx]
    M = np.vstack([np.asarray(sub[(ot["sample"]==x).values].sum(axis=0)).ravel() for x in ok])
    meta = ot.groupby("sample", observed=True)[["cond","tu","mouse","cart"]].first().loc[ok]
    meta["n_cells"] = n.loc[ok]
    return pd.DataFrame(M.astype(np.int64), index=ok, columns=adata.var_names), meta

PB, META = {}, {}
for pop, mk in [("TAM", adata.obs.TAM_global.values), ("M2", adata.obs.is_M2.values),
                ("TAM_alice", adata.obs.TAM_alice.values)]:
    PB[pop], META[pop] = pseudobulk(mk)
    print(f"{pop:<10s}: {len(PB[pop])} sample pseudobulks | "
          f"median library {PB[pop].sum(axis=1).median():,.0f} counts | "
          f"dropped {29-len(PB[pop])} samples below {MIN_CELLS} cells")

TAM       : 29 sample pseudobulks | median library 4,446,923 counts | dropped 0 samples below 20 cells


M2        : 29 sample pseudobulks | median library 7,919,148 counts | dropped 0 samples below 20 cells


TAM_alice : 29 sample pseudobulks | median library 12,115,832 counts | dropped 0 samples below 20 cells


## 4. The enrichment engine — classic weighted-KS GSEA (NES + FDR q)

Lifted from `01c` §4d, where it was validated against `gseapy.prerank` (ES to 1e-15, NES to
±0.016, q to ±0.026, zero disagreements at FDR<0.05 over 66 sets). Dependency-free so it runs on
the HPC kernel. Panel = all 50 HALLMARK sets as an honest statistical backdrop, plus the curated
metabolic GO/Reactome focus sets.

In [4]:
def read_gmt(p):
    out={}
    for line in open(p):
        x=line.rstrip("\n").split("\t")
        if len(x)>=3: out[x[0]]=x[2:]
    return out
gene_sets={}
for f in sorted(GMT_DIR.glob("*.gmt")): gene_sets.update(read_gmt(f))

METAB_FOCUS = ["GOBP_FATTY_ACID_BETA_OXIDATION","GOBP_LIPID_DROPLET_ORGANIZATION",
 "GOBP_CELLULAR_RESPONSE_TO_OXIDISED_LOW_DENSITY_LIPOPROTEIN_PARTICLE_STIMULUS",
 "GOBP_LIPID_OXIDATION","GOBP_CHOLESTEROL_EFFLUX","GOBP_CHOLESTEROL_STORAGE",
 "GOBP_CHOLESTEROL_IMPORT","REACTOME_CHOLESTEROL_BIOSYNTHESIS",
 "GOBP_LONG_CHAIN_FATTY_ACID_IMPORT_INTO_CELL","GOBP_LACTATE_METABOLIC_PROCESS",
 "GOBP_ADENOSINE_METABOLIC_PROCESS","GOBP_GLUTAMINE_METABOLIC_PROCESS",
 "GOBP_MACROPHAGE_DERIVED_FOAM_CELL_DIFFERENTIATION",
 "REACTOME_BINDING_AND_UPTAKE_OF_LIGANDS_BY_SCAVENGER_RECEPTORS",
 "GOBP_PHAGOCYTOSIS","GOBP_LYSOSOMAL_TRANSPORT","GOBP_FATTY_ACID_TRANSPORT",
 "GOBP_FATTY_ACID_CATABOLIC_PROCESS","GOBP_LIPOPHAGY","GOBP_EFFEROCYTOSIS"]
panel = {k:v for k,v in gene_sets.items() if k.startswith("HALLMARK_")}
for g in METAB_FOCUS:
    if g in gene_sets: panel[g]=gene_sets[g]
    else: print("missing from GMT:", g)
METAB = set(METAB_FOCUS) | {"HALLMARK_FATTY_ACID_METABOLISM","HALLMARK_OXIDATIVE_PHOSPHORYLATION",
    "HALLMARK_ADIPOGENESIS","HALLMARK_GLYCOLYSIS","HALLMARK_CHOLESTEROL_HOMEOSTASIS",
    "HALLMARK_PEROXISOME","HALLMARK_BILE_ACID_METABOLISM","HALLMARK_XENOBIOTIC_METABOLISM"}
print(f"panel: {len(panel)} sets ({sum(k in METAB for k in panel)} tagged metabolic)")

def _es_at_hits(pos, wcum, Wtot, N, n):
    i = np.arange(1, n+1)
    after  = wcum/Wtot
    before = np.concatenate([np.zeros(pos.shape[:-1]+(1,)), wcum[...,:-1]], axis=-1)/Wtot
    dev = np.concatenate([after-(pos+1-i)/(N-n), before-(pos-i+1)/(N-n)], axis=-1)
    hi, lo = np.maximum(dev.max(axis=-1),0.0), np.minimum(dev.min(axis=-1),0.0)
    return np.where(hi >= -lo, hi, lo)

def gsea_prerank(ranking, gene_sets, n_perm=N_PERM, weight=1.0, seed=SEED,
                 min_size=5, max_size=500, chunk=256):
    r = ranking.dropna().sort_values(ascending=False)
    w, N = np.abs(r.to_numpy(float))**weight, len(r)
    idx = pd.Series(np.arange(N), index=r.index); rng = np.random.default_rng(seed)
    rows, null_nes = [], []
    for name, members in gene_sets.items():
        m = idx.index.intersection(pd.Index(sorted(set(members)))); n = len(m)
        if not (min_size <= n <= max_size): continue
        pos = np.sort(idx.loc[m].to_numpy())
        es  = float(_es_at_hits(pos, np.cumsum(w[pos]), w[pos].sum(), N, n))
        null, done = np.empty(n_perm), 0
        while done < n_perm:
            b = min(chunk, n_perm-done)
            perm = np.sort(rng.random((b,N)).argpartition(n,axis=1)[:,:n], axis=1)
            null[done:done+b] = _es_at_hits(perm, np.cumsum(w[perm],axis=1),
                                            w[perm].sum(axis=1)[:,None], N, n)
            done += b
        pn, nn_ = null[null>=0], null[null<0]
        if es >= 0: nes_v = es/pn.mean();  p = (1+(pn>=es).sum())/(1+pn.size)
        else:       nes_v = -es/nn_.mean(); p = (1+(nn_<=es).sum())/(1+nn_.size)
        rows.append({"pathway":name,"n_genes":n,"ES":es,"NES":float(nes_v),"pval":float(p)})
        null_nes.append(np.where(null>=0, null/pn.mean(), -null/nn_.mean()))
    out = pd.DataFrame(rows)
    out["FDR_BH"] = multipletests(out["pval"], method="fdr_bh")[1]
    pool, ob = np.concatenate(null_nes), out["NES"].to_numpy()
    q = np.empty(len(ob))
    for i,v in enumerate(ob):
        if v>=0: q[i] = (pool[pool>=0]>=v).mean()/max((ob[ob>=0]>=v).mean(),1e-12)
        else:    q[i] = (pool[pool<0]<=v).mean()/max((ob[ob<0]<=v).mean(),1e-12)
    out["FDR"] = np.minimum(q,1.0)
    out["metabolic"] = out.pathway.isin(METAB)
    return out.sort_values("NES", ascending=False).reset_index(drop=True)
print("GSEA engine ready")

missing from GMT: GOBP_EFFEROCYTOSIS
panel: 69 sets (27 tagged metabolic)
GSEA engine ready


## 5. The contrast battery

**Tier A — RT, site-resolved.** The question 09 could not answer because it collapsed site into a
covariate. `Tu1` is irradiated, `Tu2` abscopal; each is tested against its own CAR-T-matched
non-RT arm at the same site.

**Tier B — within-mouse, paired.** `Tu1 − Tu2` with `~ mouse + tu`, so each animal is its own
control. Run inside the **non-RT** arms too: that is the baseline site difference with no beam,
and it is the control that makes tier B interpretable.

**Tier C — difference-in-differences.** RT's effect *on the Tu1−Tu2 gap*, formed as
`(Tu1−Tu2)_RT − (Tu1−Tu2)_non-RT` from the tier-B log-fold-changes. Taken as a difference of
shrunken LFCs rather than a DESeq2 interaction term, so the arithmetic is transparent; no per-gene
p-value is claimed for it, and it is judged by GSEA plus cross-background agreement.

**Tier D — the CAR axis.** PSCA vs TAG72. `rtAdvantage/02b` showed this is the dominant axis of
**cell-type composition** (how many of each cell type a sample contains) — that is a different
measurement from the gene expression tested here, so tier D is an independent test of whether the
same axis dominates *expression*, not a restatement of the composition result.

**Tier E — detection check only.** Treated vs `NoTx`. Not a biological comparator.

In [5]:
def run_deseq(cdf, meta, design, contrast, label):
    keep = ((cdf>0).sum(axis=0) >= MIN_SAMPLES) & (cdf.sum(axis=0) >= MIN_COUNT*len(cdf))
    X = cdf.loc[:, keep]
    dds = DeseqDataSet(counts=X, metadata=meta.loc[X.index], design=design, quiet=True)
    dds.deseq2()
    st = DeseqStats(dds, contrast=contrast, quiet=True); st.summary()
    res = st.results_df.sort_values("pvalue")
    res.attrs["label"] = label
    return res, X.shape[1]

def subset(pop, cond=None, tu=None):
    m = META[pop].copy()
    if cond is not None: m = m[m.cond.isin(cond)]
    if tu   is not None: m = m[m.tu.isin(tu)]
    return PB[pop].loc[m.index], m

PAIRS = [("PSCA","RTCyPSCA","CyPSCA"), ("TAG72","RTCyTAG72","CyTAG72")]
DE = {}
def add(key, pop, cdf, meta, design, contrast, label):
    try:
        res, ng = run_deseq(cdf, meta, design, contrast, label)
        DE[key] = res
        n = int((res.padj < 0.05).sum())
        top = ", ".join(res.index[:5]) if n else ""
        print(f"  {label:<44s} n={len(meta):2d}  {ng:,} genes -> {n:>5,} padj<0.05  {top}")
    except Exception as e:
        print(f"  {label:<44s} FAILED: {type(e).__name__}: {str(e)[:90]}")

POP = "TAM"
print("=== TIER A — RT, site-resolved ===")
for bg, rt, ct in PAIRS:
    for site, nm in [("Tu1","irradiated"), ("Tu2","abscopal")]:
        c, m = subset(POP, [rt,ct], [site])
        m = m.copy(); m["cond"] = pd.Categorical(m.cond, [ct, rt])
        add(f"A_{nm}_{bg}", POP, c, m, "~ cond", ["cond",rt,ct], f"A {nm:<10s} {bg}: {rt} vs {ct}")

print("\n=== TIER B — within-mouse Tu1(irradiated) vs Tu2(abscopal), paired ===")
for arm in ["RTCyPSCA","RTCyTAG72","CyPSCA","CyTAG72"]:
    c, m = subset(POP, [arm])
    both = m.groupby("mouse").tu.nunique(); ok = both[both==2].index
    m = m[m.mouse.isin(ok)]; c = c.loc[m.index]
    if len(m) < 4: print(f"  {arm}: only {len(m)} paired samples — skipped"); continue
    m = m.copy(); m["tu"] = pd.Categorical(m.tu, ["Tu2","Tu1"]); m["mouse"] = m.mouse.astype(str)
    tag = "RT" if arm.startswith("RT") else "control"
    add(f"B_pair_{arm}", POP, c, m, "~ mouse + tu", ["tu","Tu1","Tu2"],
        f"B pair {arm:<10s} ({tag}) Tu1 vs Tu2")

print("\n=== TIER D — the CAR axis (dominant for COMPOSITION in 02b; tested here on EXPRESSION) ===")
for lbl, a, b in [("nonRT","CyPSCA","CyTAG72"), ("RT","RTCyPSCA","RTCyTAG72")]:
    c, m = subset(POP, [a,b]); m = m.copy(); m["cond"] = pd.Categorical(m.cond, [b,a])
    add(f"D_car_{lbl}", POP, c, m, "~ tu + cond", ["cond",a,b], f"D CAR axis {lbl:<7s}: {a} vs {b}")

print("\n=== TIER E — detection check (NoTx is NOT a biological comparator) ===")
for bg, arm in [("PSCA","CyPSCA"), ("TAG72","CyTAG72")]:
    c, m = subset(POP, [arm,"NoTx"]); m = m.copy(); m["cond"]=pd.Categorical(m.cond,["NoTx",arm])
    add(f"E_tx_{bg}", POP, c, m, "~ tu + cond", ["cond",arm,"NoTx"], f"E {arm} vs NoTx")

for k,v in DE.items(): v.to_csv(OUT/"de"/f"deseq_{POP}_{k}.csv")
print(f"\n{len(DE)} DE tables -> {OUT/'de'}")

=== TIER A — RT, site-resolved ===


  A irradiated PSCA: RTCyPSCA vs CyPSCA        n= 6  10,547 genes ->    19 padj<0.05  Mmp9, Hbb-bs, Hivep2, Gzmf, Fbxo32


  A abscopal   PSCA: RTCyPSCA vs CyPSCA        n= 6  9,806 genes ->     1 padj<0.05  C6, Cd163, Cp, Tmem176a, Lrg1


  A irradiated TAG72: RTCyTAG72 vs CyTAG72     n= 5  12,005 genes ->     1 padj<0.05  Lyz1, Erdr1, Lyve1, Il1r2, Mid1


  A abscopal   TAG72: RTCyTAG72 vs CyTAG72     n= 6  11,260 genes ->     6 padj<0.05  Gdf15, Slc15a2, Gm15726, Erdr1, Dusp2

=== TIER B — within-mouse Tu1(irradiated) vs Tu2(abscopal), paired ===


...

  D CAR axis nonRT  : CyPSCA vs CyTAG72        n=11  11,057 genes ->   342 padj<0.05  Tgtp1, Pla2g16, Irf1, Socs1, Rmdn3


  D CAR axis RT     : RTCyPSCA vs RTCyTAG72    n=12  11,116 genes ->   192 padj<0.05  Gbp2b, Gbp5, Gbp2, Socs1, Tgtp1

=== TIER E — detection check (NoTx is NOT a biological comparator) ===


  E CyPSCA vs NoTx                             n=12  9,440 genes -> 2,683 padj<0.05  Mndal, Ly6e, Fn1, Zbp1, Irf7


  E CyTAG72 vs NoTx                            n=11  10,832 genes -> 4,237 padj<0.05  Bhlhe41, Ly6e, Oasl2, Tmcc3, Epsti1



12 DE tables -> /Users/janzules/Roselab/Spatial/CAR_T/results/spp1_analysis/sc_spp1tam_conditions/de


### 5b. Tier C — the difference-in-differences

`(Tu1−Tu2)_RT − (Tu1−Tu2)_non-RT`, per CAR-T background. This is the contrast that found real
structure at the *composition* level (`rtAdvantage/01`, coarse r = +0.951); tier C asks whether
the same is true of SPP1-TAM *expression*.

In [6]:
DID = {}
for bg, rt, ct in PAIRS:
    kr, kc = f"B_pair_{rt}", f"B_pair_{ct}"
    if kr not in DE or kc not in DE:
        print(f"  {bg}: missing a paired arm — DiD not formed"); continue
    a, b = DE[kr], DE[kc]
    g = a.index.intersection(b.index)
    d = pd.DataFrame({"lfc_RT": a.loc[g,"log2FoldChange"], "lfc_noRT": b.loc[g,"log2FoldChange"]})
    d["did"] = d.lfc_RT - d.lfc_noRT
    DID[bg] = d.dropna().sort_values("did", ascending=False)
    DID[bg].to_csv(OUT/"de"/f"did_{POP}_{bg}.csv")
    print(f"  {bg}: DiD over {len(DID[bg]):,} genes | "
          f"sd(did) {DID[bg].did.std():.3f} | top up: {', '.join(DID[bg].index[:4])}")
if len(DID)==2:
    g = DID["PSCA"].index.intersection(DID["TAG72"].index)
    r = stats.pearsonr(DID["PSCA"].loc[g,"did"], DID["TAG72"].loc[g,"did"])
    print(f"\n  cross-background concordance of the DiD vector: r = {r[0]:+.3f}  ({len(g):,} genes)")
    print("  (a shared RT-on-the-site-gap effect would give r clearly > 0)")

  PSCA: DiD over 9,931 genes | sd(did) 0.638 | top up: Mmp9, Snhg9, Gm42418, Slc2a6
  TAG72: DiD over 11,496 genes | sd(did) 0.357 | top up: Platr7, 1700047A11Rik, Frmpd4, Hormad1

  cross-background concordance of the DiD vector: r = +0.229  (9,923 genes)
  (a shared RT-on-the-site-gap effect would give r clearly > 0)


## 6. The descriptive view — what programs are active in each condition?

Contrasts answer *"what changed"*. The user's question also asks *"what are they doing"*, which is
a profile, not a difference. Per-sample pathway activity is scored on the pseudobulk (log-CPM,
z-scored across samples so each pathway is on a common scale), then averaged per arm × site.
This is descriptive — no test — and is the right panel for "SPP1-TAM behaviour by condition".

In [7]:
cdf, meta = PB[POP], META[POP]
cpm = np.log2(cdf.div(cdf.sum(axis=1), axis=0)*1e6 + 1)
z   = (cpm - cpm.mean(0))/(cpm.std(0)+1e-9)
prof = {}
for name, members in panel.items():
    g = z.columns.intersection(pd.Index(sorted(set(members))))
    if len(g) >= 10: prof[name] = z[g].mean(axis=1)
P = pd.DataFrame(prof).join(meta[["cond","tu","cart"]])
P["group"] = P.cond + "_" + P.tu
G = P.groupby("group", observed=True)[list(prof)].mean()
ORDER = ["NoTx_Tu1","NoTx_Tu2","CyPSCA_Tu1","CyPSCA_Tu2","RTCyPSCA_Tu1","RTCyPSCA_Tu2",
         "CyTAG72_Tu1","CyTAG72_Tu2","RTCyTAG72_Tu1","RTCyTAG72_Tu2"]
G = G.reindex([g for g in ORDER if g in G.index])
G.T.to_csv(OUT/"gsea"/f"pathway_profile_by_group_{POP}.csv")

spread = (G.max()-G.min()).sort_values(ascending=False)
print("=== pathways that DISCRIMINATE most between condition x site (range of group means) ===")
show = spread.head(18).index
print(G[show].T.round(2).to_string())
print("\n  top discriminating pathways:")
for p in spread.head(10).index:
    hi, lo = G[p].idxmax(), G[p].idxmin()
    print(f"    {p:<58s} range {spread[p]:.2f}   high {hi:<15s} low {lo}")

=== pathways that DISCRIMINATE most between condition x site (range of group means) ===
group                                                                         NoTx_Tu1  NoTx_Tu2  CyPSCA_Tu1  CyPSCA_Tu2  RTCyPSCA_Tu1  RTCyPSCA_Tu2  CyTAG72_Tu1  CyTAG72_Tu2  RTCyTAG72_Tu1  RTCyTAG72_Tu2
HALLMARK_INTERFERON_ALPHA_RESPONSE                                               -1.25     -1.19        0.53        0.29          0.57          0.39         0.25         0.27           0.23          -0.01
HALLMARK_INTERFERON_GAMMA_RESPONSE                                               -0.97     -1.04        0.59        0.31          0.53          0.41         0.09         0.01           0.16          -0.06
HALLMARK_MYC_TARGETS_V1                                                           0.86      0.74       -0.34       -0.23         -0.48         -0.21        -0.20        -0.04          -0.21           0.04
HALLMARK_E2F_TARGETS                                                              0.80      

## 7. GSEA on every contrast

Each contrast's genes are ranked by the signed DESeq2 Wald statistic (the DiD by its LFC
difference) and scored with the weighted-KS engine from §4.

In [8]:
RANK = {k: v["stat"].dropna() for k,v in DE.items()}
for bg, d in DID.items(): RANK[f"C_did_{bg}"] = d["did"]

GS = {}
for k, r in RANK.items():
    GS[k] = gsea_prerank(r, panel)
    GS[k].to_csv(OUT/"gsea"/f"gsea_{POP}_{k}.csv", index=False)
    sig_n = int((GS[k].FDR < 0.05).sum())
    top = GS[k].iloc[0]
    print(f"  {k:<22s} {sig_n:>3d} sets FDR<0.05 | top NES {top.NES:+.2f} ({top.pathway[:46]})")

NES = pd.DataFrame({k: v.set_index("pathway")["NES"] for k,v in GS.items()})
FDR = pd.DataFrame({k: v.set_index("pathway")["FDR"] for k,v in GS.items()})
NES.to_csv(OUT/"gsea"/f"nes_matrix_{POP}.csv"); FDR.to_csv(OUT/"gsea"/f"fdr_matrix_{POP}.csv")
print(f"\nNES matrix: {NES.shape[0]} pathways x {NES.shape[1]} contrasts -> {OUT/'gsea'}")

  A_irradiated_PSCA       15 sets FDR<0.05 | top NES +2.30 (HALLMARK_OXIDATIVE_PHOSPHORYLATION)


  A_abscopal_PSCA         16 sets FDR<0.05 | top NES +2.26 (HALLMARK_UV_RESPONSE_DN)


  A_irradiated_TAG72      22 sets FDR<0.05 | top NES +2.97 (HALLMARK_TNFA_SIGNALING_VIA_NFKB)


  A_abscopal_TAG72        29 sets FDR<0.05 | top NES +3.23 (HALLMARK_TNFA_SIGNALING_VIA_NFKB)


  B_pair_RTCyPSCA         29 sets FDR<0.05 | top NES +2.26 (HALLMARK_INTERFERON_ALPHA_RESPONSE)


...

  D_car_RT                14 sets FDR<0.05 | top NES +3.24 (HALLMARK_INTERFERON_GAMMA_RESPONSE)


  E_tx_PSCA               24 sets FDR<0.05 | top NES +3.21 (HALLMARK_INTERFERON_GAMMA_RESPONSE)


  E_tx_TAG72              28 sets FDR<0.05 | top NES +3.09 (HALLMARK_INTERFERON_ALPHA_RESPONSE)


  C_did_PSCA              13 sets FDR<0.05 | top NES +2.68 (HALLMARK_OXIDATIVE_PHOSPHORYLATION)


  C_did_TAG72              6 sets FDR<0.05 | top NES +2.26 (HALLMARK_INTERFERON_ALPHA_RESPONSE)

NES matrix: 69 pathways x 14 contrasts -> /Users/janzules/Roselab/Spatial/CAR_T/results/spp1_analysis/sc_spp1tam_conditions/gsea


## 8. Reproducibility — does anything agree across the two CAR-T backgrounds?

This is the only evidence standard the design supports. For each contrast family, the two
backgrounds' whole NES vectors are correlated, and pathways are counted as reproducible only if
they clear FDR<0.05 **in both** with the same sign.

In [9]:
FAMILIES = [("A_irradiated","A_irradiated_PSCA","A_irradiated_TAG72","RT at the irradiated site"),
            ("A_abscopal","A_abscopal_PSCA","A_abscopal_TAG72","RT at the abscopal site"),
            ("B_pair_RT","B_pair_RTCyPSCA","B_pair_RTCyTAG72","within-mouse Tu1-Tu2, RT arms"),
            ("B_pair_ctl","B_pair_CyPSCA","B_pair_CyTAG72","within-mouse Tu1-Tu2, NON-RT (control)"),
            ("C_did","C_did_PSCA","C_did_TAG72","difference-in-differences"),
            ("E_tx","E_tx_PSCA","E_tx_TAG72","treated vs NoTx (DETECTION CHECK)")]
rows=[]
for fam, a, b, desc in FAMILIES:
    if a not in NES or b not in NES: 
        print(f"  {desc:<42s} unavailable"); continue
    ok = NES[a].notna() & NES[b].notna()
    r = stats.pearsonr(NES.loc[ok,a], NES.loc[ok,b])[0]
    both = ((FDR[a]<0.05)&(FDR[b]<0.05)&(np.sign(NES[a])==np.sign(NES[b])))
    rows.append(dict(family=fam, description=desc, r_NES=r, n_sets=int(ok.sum()),
                     n_repro=int(both.sum()),
                     repro=", ".join(NES.index[both][:4])))
R = pd.DataFrame(rows)
print(R[["description","r_NES","n_repro","repro"]].to_string(index=False, float_format=lambda x:f"{x:+.3f}"))
R.to_csv(OUT/"gsea"/f"cross_background_reproducibility_{POP}.csv", index=False)
print("""
  r_NES is the correlation of the two backgrounds' whole NES vectors.
  The DETECTION CHECK row is the scale a real, shared perturbation produces in this pipeline.
  Any RT row far below it is a null, not a subtle effect.""")

                           description  r_NES  n_repro                                                                                                                          repro
             RT at the irradiated site -0.285        1                                                                  REACTOME_BINDING_AND_UPTAKE_OF_LIGANDS_BY_SCAVENGER_RECEPTORS
               RT at the abscopal site +0.163        3                                       HALLMARK_INTERFERON_ALPHA_RESPONSE, HALLMARK_TGF_BETA_SIGNALING, HALLMARK_UV_RESPONSE_DN
         within-mouse Tu1-Tu2, RT arms +0.542       15                   GOBP_LONG_CHAIN_FATTY_ACID_IMPORT_INTO_CELL, HALLMARK_ANGIOGENESIS, HALLMARK_APOPTOSIS, HALLMARK_COAGULATION
within-mouse Tu1-Tu2, NON-RT (control) +0.403       12                               GOBP_LACTATE_METABOLIC_PROCESS, HALLMARK_ANGIOGENESIS, HALLMARK_COAGULATION, HALLMARK_COMPLEMENT
             difference-in-differences +0.019        5 HALLMARK_ANGIOGENESIS, HALLMARK_EPI

### 8b. Leverage and definition-robustness

Two things could manufacture any positive above: a single thin pseudobulk (the smallest SPP1-TAM
sample has ~26 cells), and the signature-based population definition. Both are checked.

In [10]:
print("=== leverage: drop the thinnest sample from each arm and refit the best family ===")
best = R.loc[R[R.family.str.startswith(("A_","B_","C_"))].r_NES.idxmax()] if len(R) else None
print(f"  strongest RT-related family: {best.description} (r_NES {best.r_NES:+.3f})" if best is not None else "  none")
thin = META[POP].nsmallest(3, "n_cells")[["cond","tu","n_cells"]]
print("  thinnest pseudobulks:"); print(thin.to_string())

print("\n=== definition-robustness: repeat tier A on M2_all and on Alice's Spp1_pos_mac ===")
for pop2 in ["M2","TAM_alice"]:
    for bg, rt, ct in PAIRS:
        for site, nm in [("Tu1","irradiated"), ("Tu2","abscopal")]:
            m = META[pop2]; m = m[m.cond.isin([rt,ct]) & m.tu.isin([site])]
            if len(m) < 4: continue
            c = PB[pop2].loc[m.index]; m = m.copy(); m["cond"]=pd.Categorical(m.cond,[ct,rt])
            try:
                res,_ = run_deseq(c, m, "~ cond", ["cond",rt,ct], "")
                print(f"  {pop2:<10s} {nm:<10s} {bg:<6s}: {int((res.padj<0.05).sum()):>4,} genes padj<0.05")
            except Exception as e:
                print(f"  {pop2:<10s} {nm:<10s} {bg:<6s}: FAILED {type(e).__name__}")

=== leverage: drop the thinnest sample from each arm and refit the best family ===
  strongest RT-related family: within-mouse Tu1-Tu2, RT arms (r_NES +0.542)
  thinnest pseudobulks:
                cond   tu  n_cells
sample                            
CyPSCA_Tu2_3  CyPSCA  Tu2       26
NoTx_Tu2_3      NoTx  Tu2       36
NoTx_Tu1_2      NoTx  Tu1       89

=== definition-robustness: repeat tier A on M2_all and on Alice's Spp1_pos_mac ===


  M2         irradiated PSCA  :   36 genes padj<0.05


  M2         abscopal   PSCA  :    1 genes padj<0.05


  M2         irradiated TAG72 :    0 genes padj<0.05


  M2         abscopal   TAG72 :    4 genes padj<0.05


  TAM_alice  irradiated PSCA  :   35 genes padj<0.05


  TAM_alice  abscopal   PSCA  :    0 genes padj<0.05


  TAM_alice  irradiated TAG72 :    4 genes padj<0.05


  TAM_alice  abscopal   TAG72 :    4 genes padj<0.05


## 9. SUMMARY — report these back

In [11]:
print("="*94); print("SUMMARY — SPP1-TAM behaviour by condition (pseudobulk DE -> GSEA)"); print("="*94)
de_counts = {k: int((v.padj<0.05).sum()) for k,v in DE.items()}
print("\nDE hits (padj<0.05) per contrast:")
for k,v in sorted(de_counts.items(), key=lambda x:-x[1]): print(f"   {k:<24s} {v:>6,}")
print("\nCross-background reproducibility of the GSEA NES vectors:")
for _,row in R.iterrows(): print(f"   {row.description:<44s} r = {row.r_NES:+.3f}   both-FDR<0.05: {row.n_repro}")
print(f"""
DESIGN NOTES THAT MUST TRAVEL WITH ANY CLAIM
  * No RT-alone arm -> every RT contrast is 'adding RT on top of Cy + CAR-T'.
  * Site-resolved contrasts are 3 vs 3 (3 vs 2 for CyTAG72_Tu1). Evidence = sign agreement
    across the two CAR-T backgrounds, never a single p-value.
  * NoTx is a DETECTION CHECK only (tumour 25-42% of cells vs 4-6% in treated arms).
  * rtAdvantage/02b found WHICH CAR dominates CELL-TYPE COMPOSITION -- CLR abundance of 18
    cell_type_lvl2 types over 23 treated samples, OLS ~cart+RT+site+cart:RT, % of type-II SS,
    averaged over cell types: cart 10.3%, cart:RT 8.7%, site 5.3%, RT 3.5%, and residual
    (mouse-to-mouse) 72.3% -- i.e. CAR is the largest ASSIGNABLE term, not most of the variance.
    PERMANOVA (Aitchison, treated only) cart R2=0.103 p=0.021, RT R2=0.037 p=0.597.
    That is COMPOSITION. Tier D here tests EXPRESSION, a separate modality -- so tier D's
    342/192 genes INDEPENDENTLY confirm the CAR axis rather than restating 02b.
  * Enrichment is GSEA NES, not the rank-sum z used in 09 -- the z is anti-conservative and
    missed the top hit in 01c.

FILES -> {OUT}
  de/deseq_TAM_<contrast>.csv, de/did_TAM_<bg>.csv
  gsea/gsea_TAM_<contrast>.csv, gsea/nes_matrix_TAM.csv, gsea/fdr_matrix_TAM.csv
  gsea/pathway_profile_by_group_TAM.csv, gsea/cross_background_reproducibility_TAM.csv
""")
print("="*94)

SUMMARY — SPP1-TAM behaviour by condition (pseudobulk DE -> GSEA)

DE hits (padj<0.05) per contrast:
   E_tx_TAG72                4,237
   E_tx_PSCA                 2,683
   D_car_nonRT                 342
   D_car_RT                    192
   B_pair_RTCyTAG72            129
   B_pair_RTCyPSCA              40
   A_irradiated_PSCA            19
   A_abscopal_TAG72              6
   B_pair_CyPSCA                 6
   B_pair_CyTAG72                4
   A_abscopal_PSCA               1
   A_irradiated_TAG72            1

Cross-background reproducibility of the GSEA NES vectors:
   RT at the irradiated site                    r = -0.285   both-FDR<0.05: 1
   RT at the abscopal site                      r = +0.163   both-FDR<0.05: 3
   within-mouse Tu1-Tu2, RT arms                r = +0.542   both-FDR<0.05: 15
   within-mouse Tu1-Tu2, NON-RT (control)       r = +0.403   both-FDR<0.05: 12
   difference-in-differences                    r = +0.019   both-FDR<0.05: 5
   treated vs NoTx (DETECTIO